# 실습 10. 반복측정 자료 분석

사회약학 연구방법 노트의 '실습 10. 반복측정 자료 분석'에 딸린 노트북입니다. 앞부분(a–c)은 lme4 패키지의 sleepstudy 자료(수면 제한 연구 참가자 18명의 10일 반응시간)로 자료구조를 바꾸고 반복측정 분산분석과 선형 혼합모형을 적합합니다. 뒷부분(d)은 MASS 패키지의 epil 자료(뇌전증 임상시험 59명의 2주 발작 건수 네 번)로 포아송 GEE를 적합합니다. 셀을 위에서부터 차례로 실행하세요. 설명과 출력 읽는 법은 사이트의 실습 10에 있습니다.


## a. 실습 데이터와 자료구조 변환

이 실습이 쓰는 패키지를 불러와 버전을 확인한 뒤, sleepstudy와 epil 자료를 인터넷에서 읽어 옵니다. 인터넷 연결이 필요합니다.

### 셀 1. 패키지 불러오기와 버전 확인

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib, scipy, statsmodels

print("pandas      ", pd.__version__)
print("numpy       ", np.__version__)
print("matplotlib  ", matplotlib.__version__)
print("scipy       ", scipy.__version__)
print("statsmodels ", statsmodels.__version__)

### 셀 2. sleepstudy 불러오기 (긴 형식)

In [ ]:
url = ("https://vincentarelbundock.github.io/Rdatasets/csv/"
       "lme4/sleepstudy.csv")
sleep = pd.read_csv(url).drop(columns="rownames")
print(sleep.shape)
sleep.head(12)

### 셀 3. 몇 명을 몇 번 측정했나

In [ ]:
print(sleep["Subject"].nunique())            # 참가자 수
print(sleep.groupby("Subject").size().value_counts())
print(sleep["Days"].unique())                # 측정 시점

### 셀 4. 긴 형식 → 넓은 형식 (pivot)

In [ ]:
wide = sleep.pivot(index="Subject", columns="Days",
                   values="Reaction")
print(wide.shape)
wide.round(0).head()

### 셀 5. 넓은 형식 → 긴 형식 (melt)

In [ ]:
long = wide.reset_index().melt(id_vars="Subject",
                               var_name="Days",
                               value_name="Reaction")
long = long.sort_values(["Subject", "Days"])
print(long.shape)
long.head()

### 셀 6. 스파게티 그림

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for sid, g in sleep.groupby("Subject"):      # 참가자마다 선 하나
    ax.plot(g["Days"], g["Reaction"], color="grey", alpha=0.5)
mean = sleep.groupby("Days")["Reaction"].mean()
ax.plot(mean.index, mean.values, color="C0", lw=3,
        label="Mean of 18 subjects")
ax.set_xlabel("Days of sleep deprivation")
ax.set_ylabel("Average reaction time (ms)")
ax.legend()
plt.show()

### 셀 7. epil 불러오기

In [ ]:
url = ("https://vincentarelbundock.github.io/Rdatasets/csv/"
       "MASS/epil.csv")
epil = pd.read_csv(url).drop(columns="rownames")
epil["trt"] = pd.Categorical(epil["trt"],
                             categories=["placebo", "progabide"])
print(epil.shape, epil["subject"].nunique())
epil.head(8)

## b. 반복측정 분산분석

### 셀 8. 반복측정 분산분석 (AnovaRM)

In [ ]:
from statsmodels.stats.anova import AnovaRM

aov = AnovaRM(sleep, depvar="Reaction", subject="Subject",
              within=["Days"]).fit()
print(aov)

### 셀 9. Greenhouse–Geisser ε 직접 계산

In [ ]:
from scipy import stats

S = np.cov(wide.values, rowvar=False)   # 10×10 공분산 행렬
k, n = S.shape[0], wide.shape[0]        # 시점 10, 참가자 18
# 이중 중심화: 행 평균·열 평균을 빼고 전체 평균을 더함
Sc = S - S.mean(axis=0) - S.mean(axis=1)[:, None] + S.mean()
eps = np.trace(Sc)**2 / ((k - 1) * np.sum(Sc**2))
print("GG epsilon:", round(eps, 3))

F = aov.anova_table.loc["Days", "F Value"]
df1, df2 = k - 1, (k - 1) * (n - 1)
print("보정 자유도:", round(eps * df1, 2), round(eps * df2, 2))
print("p, 보정 없음:", stats.f.sf(F, df1, df2))
print("p, GG 보정  :", stats.f.sf(F, eps * df1, eps * df2))
print("p, 하한 보정:", stats.f.sf(F, 1, n - 1))

### 셀 10. Mauchly 검정과 Huynh–Feldt ε

In [ ]:
lam = np.linalg.eigvalsh(Sc)[1:]         # 고윳값 k - 1개 (맨 앞의 0 하나는 뺌)
W = np.prod(lam) / np.mean(lam)**(k - 1)  # Mauchly의 W (1이면 구형성)
d = 1 - (2*(k - 1)**2 + (k - 1) + 2) / (6*(k - 1)*(n - 1))
chi2 = -(n - 1) * d * np.log(W)          # 카이제곱 근사
df_m = k * (k - 1) // 2 - 1
print("Mauchly W:", round(W, 6), " chi2:", round(chi2, 2),
      " df:", df_m, " p:", stats.chi2.sf(chi2, df_m))

hf = (n*(k - 1)*eps - 2) / ((k - 1)*(n - 1 - (k - 1)*eps))
hf = min(hf, 1)                          # 1을 넘으면 1로 둠
print("HF epsilon:", round(hf, 3))
print("p, HF 보정  :", stats.f.sf(F, hf * df1, hf * df2))

한 칸이 빠지면 오류가 나는 것을 보여 주는 셀(셀 11)은 이 노트북에서 뺐습니다(사이트의 실습 10 b 절에 있습니다).

## c. 선형 혼합모형

### 셀 12. 무작위 절편 모형

In [ ]:
import statsmodels.formula.api as smf

m1 = smf.mixedlm("Reaction ~ Days", data=sleep,
                 groups=sleep["Subject"]).fit()
print(m1.summary())

### 셀 13. ICC와 표준오차 비교

In [ ]:
tau2 = m1.cov_re.iloc[0, 0]     # 참가자 간 분산 (Group Var)
sigma2 = m1.scale               # 참가자 내 분산 (Scale)
print("ICC:", round(tau2 / (tau2 + sigma2), 3))

ols = smf.ols("Reaction ~ Days", data=sleep).fit()
print("OLS의 Days SE      :", round(ols.bse["Days"], 3))
print("무작위 절편 Days SE:", round(m1.bse_fe["Days"], 3))

### 셀 14. 무작위 절편 + 기울기 모형

In [ ]:
m2 = smf.mixedlm("Reaction ~ Days", data=sleep,
                 groups=sleep["Subject"],
                 re_formula="~Days").fit()
print(m2.summary())

### 셀 15. 우도비 검정, REML과 ML

In [ ]:
lr = 2 * (m2.llf - m1.llf)        # 두 모형 모두 REML로 적합
print("LR 통계량:", round(lr, 2))
print("p (자유도 2):", stats.chi2.sf(lr, df=2))

m2_ml = smf.mixedlm("Reaction ~ Days", data=sleep,
                    groups=sleep["Subject"],
                    re_formula="~Days").fit(reml=False)
vc = pd.DataFrame(
    {"REML": [m2.cov_re.iloc[0, 0], m2.cov_re.iloc[1, 1], m2.scale],
     "ML": [m2_ml.cov_re.iloc[0, 0], m2_ml.cov_re.iloc[1, 1],
            m2_ml.scale]},
    index=["intercept var", "slope var", "residual var"])
vc.round(1)

### 셀 16. Days 계수의 표준오차 네 가지

In [ ]:
ols_cl = smf.ols("Reaction ~ Days", data=sleep).fit(
    cov_type="cluster", cov_kwds={"groups": sleep["Subject"]})
se = pd.Series({"OLS (independent)": ols.bse["Days"],
                "OLS + cluster-robust": ols_cl.bse["Days"],
                "random intercept": m1.bse_fe["Days"],
                "random intercept + slope": m2.bse_fe["Days"]})
se.round(3).to_frame("SE of Days")

### 셀 17. 참가자별 직선과 수축

In [ ]:
fe = m2.fe_params                        # 고정효과(절편, 기울기)
fig, axes = plt.subplots(1, 3, figsize=(10, 3.4), sharey=True)
for ax, sid in zip(axes, [308, 335, 350]):
    g = sleep[sleep["Subject"] == sid]
    x = g["Days"]
    re = m2.random_effects[sid]          # 이 사람의 무작위효과
    b0 = fe["Intercept"] + re["Group"]
    b1 = fe["Days"] + re["Days"]
    own_b1, own_b0 = np.polyfit(x, g["Reaction"], 1)
    ax.scatter(x, g["Reaction"], color="grey", s=15)
    ax.plot(x, own_b0 + own_b1 * x, "--", color="C1",
            label="Own data only")
    ax.plot(x, b0 + b1 * x, color="C0", label="Mixed model")
    ax.plot(x, fe["Intercept"] + fe["Days"] * x, ":",
            color="black", label="Average")
    ax.set_title(f"Subject {sid}: slope {own_b1:.1f} -> {b1:.1f}")
    ax.set_xlabel("Days")
axes[0].set_ylabel("Reaction time (ms)")
axes[0].legend(fontsize=8)
plt.show()

## d. 일반화 추정 방정식

### 셀 18. 군별·기간별 평균 발작 건수

In [ ]:
print(epil.groupby("trt", observed=True)["subject"].nunique())
tab = epil.groupby(["trt", "period"], observed=True)["y"].mean()
tab.unstack().round(2)

### 셀 19. GEE (포아송, 교환가능 작업상관)

In [ ]:
import statsmodels.api as sm

form = "y ~ trt + np.log(base / 4) + np.log(age) + C(period)"
gee_ex = smf.gee(form, groups="subject", data=epil,
                 family=sm.families.Poisson(),
                 cov_struct=sm.cov_struct.Exchangeable()).fit()
print(gee_ex.summary())
print(gee_ex.model.cov_struct.summary())

### 셀 20. 발생률비로 바꾸기

In [ ]:
ci = gee_ex.conf_int()                 # 로그 척도의 95% CI
rr = pd.DataFrame({"RR": np.exp(gee_ex.params),
                   "2.5%": np.exp(ci[0]),
                   "97.5%": np.exp(ci[1])})
rr.round(3)

### 셀 21. 작업상관과 표준오차 비교

In [ ]:
gee_in = smf.gee(form, groups="subject", data=epil,
                 family=sm.families.Poisson(),
                 cov_struct=sm.cov_struct.Independence()).fit()
glm = smf.glm(form, data=epil, family=sm.families.Poisson()).fit()
print("GLM Pearson chi2/df:", round(glm.pearson_chi2 / glm.df_resid, 2))

t = "trt[T.progabide]"
fits = {"Poisson GLM": glm, "GEE independence": gee_in,
        "GEE exchangeable": gee_ex}
pd.DataFrame({"coef": [f.params[t] for f in fits.values()],
              "SE": [f.bse[t] for f in fits.values()]},
             index=fits.keys()).round(4)

## e. 과제 정답

**과제 1.** 셀 14의 무작위 절편 + 기울기 모형에서 Days 행을 Results 문장으로 옮기고, 이 숫자가 말해 주지 않는 것을 두 가지 이상 쓰세요. 혼합모형이 예측한 참가자별 기울기도 모두 구해 보세요.

### 셀 22. 과제 1 정답. 수면 제한 하루당 기울기 읽기

In [ ]:
slopes = pd.Series({sid: m2.fe_params["Days"] + re["Days"]
                    for sid, re in m2.random_effects.items()})
print(slopes.sort_values().round(1).to_string())   # 참가자별 기울기
print("5 ms/일보다 작은 참가자:", (slopes < 5).sum(), "명")

**과제 2.** 셀 19의 GEE에서 기저 발작 건수(np.log(base / 4))를 빼고 다시 적합하세요. progabide의 발생률비, 표준오차, 작업상관이 어떻게 바뀌는지 원래 모형과 비교하고 왜 그런지 설명하세요.

### 셀 23. 과제 2 정답. 기저 발작 건수를 보정하지 않으면

In [ ]:
form0 = "y ~ trt + np.log(age) + C(period)"     # 기저 발작 건수를 뺀 모형
gee0 = smf.gee(form0, groups="subject", data=epil,
               family=sm.families.Poisson(),
               cov_struct=sm.cov_struct.Exchangeable()).fit()
for name, g in [("with baseline", gee_ex), ("without baseline", gee0)]:
    lo, hi = np.exp(g.conf_int().loc[t])
    print(f"{name:16}: RR {np.exp(g.params[t]):.3f} ({lo:.3f}-{hi:.3f})",
          f" SE {g.bse[t]:.3f}  alpha {g.model.cov_struct.dep_params:.3f}")

**과제 3.** 동료가 Days를 범주형으로 넣은 무작위 절편 모형을 기본 설정(REML)으로 적합하고, 셀 12의 직선 모형과 로그우도를 비교해 "LR = 56.97, 자유도 8, P < 0.001이므로 반응시간은 직선으로 늘지 않는다"라고 썼습니다. 동료의 계산을 재현하고, 무엇이 왜 틀렸는지 찾아 바른 비교를 해 보세요.

### 셀 24. 과제 3 정답. 시간을 범주형으로 넣은 모형과 비교하기

In [ ]:
m_cat = smf.mixedlm("Reaction ~ C(Days)", data=sleep,
                    groups=sleep["Subject"]).fit()       # 동료의 모형(REML)
print("REML로 비교한 LR:", round(2 * (m_cat.llf - m1.llf), 2))

m_cat_ml = smf.mixedlm("Reaction ~ C(Days)", data=sleep,
                       groups=sleep["Subject"]).fit(reml=False)
m1_ml = smf.mixedlm("Reaction ~ Days", data=sleep,
                    groups=sleep["Subject"]).fit(reml=False)
lr_ml = 2 * (m_cat_ml.llf - m1_ml.llf)
print("ML로 비교한 LR  :", round(lr_ml, 2),
      " p (자유도 8):", round(stats.chi2.sf(lr_ml, 8), 3))
print("AIC (ML): linear", round(m1_ml.aic, 1),
      " categorical", round(m_cat_ml.aic, 1))
wt = m_cat.wald_test_terms(scalar=True).table   # 계수 9개를 함께 본 Wald 검정
print(f"Wald chi2 / 9 = {wt.loc['C(Days)', 'statistic'] / 9:.2f}")